# Project 52 — Local Spreadsheet Analyst Agent
## Answer Questions Over CSV/XLSX and Generate Insights

**Stack:** LangChain · Ollama · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-experimental pandas

## Step 1 — Create Sample Dataset

In [ ]:
import pandas as pd
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)

sales = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=12, freq="MS"),
    "region": ["North","South","East","West"]*3,
    "product": ["Widget A","Widget B","Widget C"]*4,
    "revenue": [12000,15000,9000,11000,13500,16000,10500,12500,14000,17000,11500,13000],
    "units": [240,300,180,220,270,320,210,250,280,340,230,260],
    "returns": [12,5,8,15,10,3,14,9,7,2,11,6],
})
sales.to_csv("sample_data/sales_data.csv", index=False)
print(sales.to_string(index=False))
print(f"\nShape: {sales.shape}")

## Step 2 — Setup Pandas Agent

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

def analyze_data(question, df):
    """Use LLM to generate and execute pandas code for analysis."""
    # Generate code
    code_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a data analyst. Given a pandas DataFrame `df` with columns:
{columns}

Sample data:
{sample}

Write Python code using pandas to answer the question.
Return ONLY the Python code, no explanation. Use print() for output."""),
        ("human", "{question}")
    ])
    code_chain = code_prompt | llm | StrOutputParser()
    generated_code = code_chain.invoke({
        "columns": str(list(df.columns)),
        "sample": df.head(3).to_string(),
        "question": question,
    })

    # Clean up code
    code_lines = []
    for line in generated_code.split("\n"):
        line = line.strip()
        if line and not line.startswith("```"):
            code_lines.append(line)
    clean_code = "\n".join(code_lines)

    # Execute safely
    import io, contextlib
    output = io.StringIO()
    try:
        with contextlib.redirect_stdout(output):
            exec(clean_code, {"df": df, "pd": pd, "print": print})
        result = output.getvalue()
    except Exception as e:
        result = f"Error: {e}\nGenerated code:\n{clean_code}"

    return {"code": clean_code, "result": result}

print("Spreadsheet analyst ready!")

## Step 3 — Ask Questions About the Data

In [ ]:
questions = [
    "What is the total revenue by region?",
    "Which product has the highest return rate?",
    "What's the average revenue per unit sold?",
    "Show the month with the highest revenue",
]

for q in questions:
    print(f"\nQ: {q}")
    analysis = analyze_data(q, sales)
    print(f"Code: {analysis['code'][:100]}...")
    print(f"Result: {analysis['result']}")

## Step 4 — Generate Insights Report

In [ ]:
insight_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a business analyst. Given this sales data, provide "
     "5 key insights and 3 actionable recommendations."),
    ("human", "Data summary:\n{data}")
])
insight_chain = insight_prompt | llm | StrOutputParser()

data_summary = f"""
Total Revenue: ${sales['revenue'].sum():,}
Avg Monthly Revenue: ${sales['revenue'].mean():,.0f}
Revenue by Region: {sales.groupby('region')['revenue'].sum().to_dict()}
Revenue by Product: {sales.groupby('product')['revenue'].sum().to_dict()}
Total Returns: {sales['returns'].sum()} ({sales['returns'].sum()/sales['units'].sum()*100:.1f}%)
Best Month: {sales.loc[sales['revenue'].idxmax(), 'date'].strftime('%B %Y')}
"""

insights = insight_chain.invoke({"data": data_summary})
print("BUSINESS INSIGHTS REPORT")
print("="*40)
print(insights)

## What You Learned
- **NL-to-pandas** code generation for data analysis
- **Safe code execution** with output capture
- **Automated insight generation** from data summaries